# 一键版 · L4 想法到视频

**使用方法：选择 L4 GPU 和高内存运行时，然后点击「运行时 → 全部运行」。**
无需修改开关、输入密码或手动安装依赖。首次安装和数十 GB 模型下载需要等待，全部完成后会显示：

**公网 HTTPS 地址 + 账号 `owner` + 自动生成的强密码。**

默认开启带鉴权网页/API 和 Cloudflare 公网隧道。网页里输入想法、点击「生成短片」，即可依次查看文案、首帧和视频。
初始参数为 33 帧、20 步，24 FPS，约 1.38 秒；跑通后可在网页改为 121 帧、50 步，约 5.04 秒。

- 目标硬件：**L4 24GB 显存、64GB 系统内存、200GB 硬盘**。
- Qwen 1.5B → SD-Turbo → 官方 Wan2.2 TI2V-5B，三阶段独立进程释放显存。
- 已内置 Xet 下载修复、SD-Turbo 重复权重筛选，以及新版公网就绪/鉴权自检。
- 安装、下载和网页待机阶段 GPU 使用率低是正常现象，提交生成任务后才运行主要推理。
- 密码每个新服务会话自动随机生成，仅保留在运行时内存和最后的输出中，不写入源代码、配置或日志。

**请妥善保存最终地址和密码；分享笔记本前清除运行输出。**
Colab/Cloudflare 服务回收或网络中断会使地址失效；不提供保活或 GPU 抢占功能。
**未在 L4/Colab 实际执行，未验证真实公网连通性、峰值显存、耗时或生成画质。**
此文件保存时不含真实账号密码或运行输出。启动过程会自动检查依赖、GPU、小算子、模型文件和访问鉴权，失败时保留具体日志。


## 1 · 自动准备运行环境
本节不需要手动修改。与普通版使用不同的服务目录；已有模型缓存可自动复用。


In [ ]:
import os, sys, json, subprocess, shutil, time, signal, base64, urllib.request
from pathlib import Path
from IPython.display import display, Image, Video, JSON
BASE = Path('/content/wan_l4_oneclick')
BASE.mkdir(parents=True, exist_ok=True)
os.environ['WAN_LAB_HOME'] = str(BASE)
ENABLE_WEB = True        # 一键版默认开启网页
ENABLE_PUBLIC = True     # 一键版默认开启公网，仍强制鉴权
IDEA = '一只金毛犬在晨光中的草地上慢慢奔跑，野花轻轻摇动，温暖电影感。'
SEED, FRAMES, STEPS = 42, 33, 20
# 正式效果可用：FRAMES, STEPS = 121, 50；请先完成默认短片试跑。
def run_logged(cmd, name, env=None, cwd=None):
    """实时展示输出并留日志；失败立即停止，避免继续运行后续阶段。"""
    with (BASE / (name + '.log')).open('w') as log:
        p = subprocess.Popen([str(x) for x in cmd], cwd=cwd, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
            bufsize=1, start_new_session=True)
        try:
            for line in p.stdout:
                log.write(line); log.flush(); print(line, end='')
            rc = p.wait()
        except BaseException:
            os.killpg(p.pid, signal.SIGTERM)
            try: p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL); p.wait()
            raise
    if rc: raise RuntimeError(f'{name} 失败，查看 {BASE / (name + ".log")}')

ONECLICK_STATE = globals().get('ONECLICK_STATE', {})
BOOT_READY = False
print('一键启动：检查硬件 → 安装 → GPU 自检 → 下载模型 → 启动网页 → 公网鉴权自检')


In [ ]:
FILES = {'core.py': '"""CPU-only orchestration; model stages always live in separate processes."""\nimport contextlib\nimport fcntl\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport time\nimport uuid\n\nBASE = Path(os.environ.get(\'WAN_LAB_HOME\', \'/content/wan_l4_lab\')).resolve()\nRUNS = BASE / \'runs\'\nPY = BASE / \'venv/bin/python\'\nREPO = BASE / \'Wan2.2\'\n\ndef load_config():\n    return json.loads((BASE / \'config.json\').read_text())\n\ndef new_run(idea, seed=42, frames=33, steps=20):\n    if not isinstance(idea, str) or not 1 <= len(idea.strip()) <= 2000:\n        raise ValueError(\'想法须为 1–2000 字符\')\n    if type(seed) is not int or not 0 <= seed <= 4294967295:\n        raise ValueError(\'seed 超出范围\')\n    if frames not in (33, 65, 121) or type(frames) is not int:\n        raise ValueError(\'frames 只能为 33、65、121（4n+1）\')\n    if type(steps) is not int or not 10 <= steps <= 50:\n        raise ValueError(\'steps 须为 10–50 的整数\')\n    run = RUNS / uuid.uuid4().hex\n    run.mkdir(parents=True)\n    (run / \'request.json\').write_text(json.dumps(dict(idea=idea.strip(), seed=seed,\n        frames=frames, steps=steps), ensure_ascii=False, indent=2))\n    return run\n\ndef read_plan(run):\n    plan = json.loads((Path(run) / \'plan.json\').read_text())\n    for key, limit in [(\'copy_zh\', 2000), (\'image_prompt\', 1800), (\'video_prompt\', 1800)]:\n        if not isinstance(plan.get(key), str) or not 1 <= len(plan[key].strip()) <= limit:\n            raise ValueError(\'文案 JSON 缺少有效字段：\' + key)\n    return plan\n\n@contextlib.contextmanager\ndef gpu_lock():\n    BASE.mkdir(parents=True, exist_ok=True)\n    with (BASE / \'gpu.lock\').open(\'a\') as handle:\n        try:\n            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError:\n            raise RuntimeError(\'GPU 正忙；请等待当前阶段完成\')\n        try:\n            yield\n        finally:\n            fcntl.flock(handle, fcntl.LOCK_UN)\n\ndef video_command(run, config=None):\n    run = Path(run).resolve()\n    c = config or load_config()\n    req = json.loads((run / \'request.json\').read_text())\n    plan = read_plan(run)\n    if not (run / \'image.png\').is_file():\n        raise FileNotFoundError(\'先生成首帧 image.png\')\n    return [str(PY), str(REPO / \'generate.py\'), \'--task\', \'ti2v-5B\',\n        \'--size\', \'1280*704\', \'--ckpt_dir\', c[\'models\'][\'wan\'],\n        \'--offload_model\', \'True\', \'--convert_model_dtype\', \'--t5_cpu\',\n        \'--image\', str(run / \'image.png\'), \'--prompt\', plan[\'video_prompt\'],\n        \'--frame_num\', str(req[\'frames\']), \'--sample_steps\', str(req[\'steps\']),\n        \'--sample_solver\', \'unipc\', \'--sample_shift\', \'5.0\',\n        \'--sample_guide_scale\', \'5.0\', \'--base_seed\', str(req[\'seed\']),\n        \'--save_file\', str(run / \'video.mp4\')]\n\ndef terminate_tree(proc):\n    if proc.poll() is None:\n        os.killpg(proc.pid, signal.SIGTERM)\n        try:\n            proc.wait(timeout=10)\n        except subprocess.TimeoutExpired:\n            os.killpg(proc.pid, signal.SIGKILL)\n            proc.wait()\n\ndef run_stage(stage, run, timeout=7200):\n    if stage not in (\'text\', \'image\', \'video\'):\n        raise ValueError(\'未知阶段\')\n    run = Path(run).resolve()\n    if run.parent != RUNS or not (run / \'request.json\').is_file():\n        raise ValueError(\'无效运行目录\')\n    with gpu_lock():\n        cmd = video_command(run) if stage == \'video\' else [str(PY),\n            str(BASE / \'worker.py\'), stage, str(run)]\n        env = dict(os.environ, PYTHONUNBUFFERED=\'1\', TOKENIZERS_PARALLELISM=\'false\',\n            HF_HUB_OFFLINE=\'1\', TRANSFORMERS_OFFLINE=\'1\',\n            PYTORCH_CUDA_ALLOC_CONF=\'expandable_segments:True\')\n        # Do not pass web credentials or optional Hub token into model processes.\n        for key in (\'LAB_PASSWORD\', \'LAB_USER\', \'HF_TOKEN\'):\n            env.pop(key, None)\n        log = run / (stage + \'.log\')\n        start = time.monotonic()\n        with log.open(\'w\') as out:\n            proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=out,\n                stderr=subprocess.STDOUT, start_new_session=True)\n            try:\n                code = proc.wait(timeout=timeout)\n            finally:\n                terminate_tree(proc)\n        if code:\n            raise RuntimeError(f\'{stage} 失败（exit={code}），查看 {log.name}\')\n        expected = {\'text\': \'plan.json\', \'image\': \'image.png\', \'video\': \'video.mp4\'}[stage]\n        if not (run / expected).is_file() or (run / expected).stat().st_size == 0:\n            raise RuntimeError(\'阶段结束但未生成文件：\' + expected)\n        if stage == \'text\':\n            read_plan(run)\n        if stage == \'video\':\n            probe = subprocess.run([\'ffprobe\', \'-v\', \'error\', \'-select_streams\', \'v:0\',\n                \'-show_entries\', \'stream=width,height,nb_frames,r_frame_rate,duration\',\n                \'-of\', \'json\', str(run / expected)], capture_output=True, text=True, check=True)\n            info = json.loads(probe.stdout)[\'streams\'][0]\n            req = json.loads((run / \'request.json\').read_text())\n            if (int(info[\'width\']), int(info[\'height\'])) != (1280, 704):\n                raise RuntimeError(\'输出尺寸不符：\' + str(info))\n            if int(info.get(\'nb_frames\', -1)) != req[\'frames\'] or info[\'r_frame_rate\'] != \'24/1\':\n                raise RuntimeError(\'输出帧数或帧率不符：\' + str(info))\n            (run / \'video_info.json\').write_text(json.dumps(info, indent=2))\n        metric = {\'stage\': stage, \'seconds\': round(time.monotonic()-start, 2),\n                  \'note\': \'子进程已退出，CUDA 上下文随进程释放；不是峰值显存测量\'}\n        try:\n            metric[\'gpu_after\'] = subprocess.check_output([\'nvidia-smi\',\n                \'--query-gpu=memory.used,memory.free\', \'--format=csv,noheader\'], text=True).strip()\n        except (OSError, subprocess.SubprocessError):\n            pass\n        (run / (stage + \'_metrics.json\')).write_text(json.dumps(metric, ensure_ascii=False, indent=2))\n        return run / expected\n', 'worker.py': "import json\nfrom pathlib import Path\nimport sys\nfrom core import BASE, load_config, read_plan\n\ndef main(stage, folder):\n    import torch\n    run = Path(folder)\n    config = load_config()\n    req = json.loads((run / 'request.json').read_text())\n    torch.manual_seed(req['seed'])\n    if stage == 'text':\n        from transformers import AutoTokenizer, AutoModelForCausalLM\n        model_dir = config['models']['text']\n        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True,\n                                                 trust_remote_code=False)\n        model = AutoModelForCausalLM.from_pretrained(model_dir,\n            torch_dtype=torch.bfloat16, local_files_only=True,\n            trust_remote_code=False, use_safetensors=True).to('cuda').eval()\n        messages = [dict(role='system', content=(\n            'You are a video creative writer. Return ONLY one valid JSON object, '\n            'no markdown. Keys: copy_zh (short Chinese promotional copy), '\n            'image_prompt (English description of ONE initial frame), '\n            'video_prompt (English description of the SAME scene with gentle motion '\n            'and camera movement, no cuts). Keep subject, setting and lighting consistent. '\n            'No subtitles or rendered text. All values must be nonempty strings.')),\n            dict(role='user', content=req['idea'])]\n        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        inputs = tokenizer(text, return_tensors='pt').to('cuda')\n        with torch.inference_mode():\n            output = model.generate(**inputs, max_new_tokens=700, do_sample=False,\n                                    pad_token_id=tokenizer.eos_token_id)\n        raw = tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)\n        (run / 'text_raw.txt').write_text(raw)\n        # No invented fallback prompts: malformed output stops with raw text saved for repair.\n        begin, end = raw.find('{'), raw.rfind('}')\n        if begin < 0 or end <= begin:\n            raise ValueError('模型没有返回 JSON，请检查 text_raw.txt 并修正 plan.json')\n        plan = json.loads(raw[begin:end+1])\n        (run / 'plan.json').write_text(json.dumps(plan, ensure_ascii=False, indent=2))\n        read_plan(run)\n        print(json.dumps(plan, ensure_ascii=False, indent=2))\n    elif stage == 'image':\n        from diffusers import AutoPipelineForText2Image\n        from PIL import Image, ImageOps\n        plan = read_plan(run)\n        pipe = AutoPipelineForText2Image.from_pretrained(config['models']['image'],\n            torch_dtype=torch.float16, variant='fp16', use_safetensors=True,\n            local_files_only=True).to('cuda')\n        pipe.enable_vae_slicing()\n        pipe.enable_vae_tiling()\n        with torch.inference_mode():\n            result = pipe(prompt=plan['image_prompt'], num_inference_steps=4,\n                guidance_scale=0.0, width=512, height=512,\n                generator=torch.Generator(device='cuda').manual_seed(req['seed']))\n        if result.nsfw_content_detected is not None and any(result.nsfw_content_detected):\n            raise ValueError('模型安全检查拦截了输出，请修改想法')\n        image = result.images[0].convert('RGB')\n        image.save(run / 'image_original.png')\n        # Display this exact crop before video generation; upscaling adds no detail.\n        ImageOps.fit(image, (1280, 704), method=Image.Resampling.LANCZOS).save(run / 'image.png')\n        print('已保存原始 512×512 图和裁切放大的 1280×704 视频首帧')\n    else:\n        raise ValueError(stage)\n\nif __name__ == '__main__':\n    main(*sys.argv[1:])\n", 'server.py': '"""Single-user demo: HTTP Basic auth guards every route, including media."""\nimport base64\nimport binascii\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport threading\nimport time\nfrom urllib.parse import urlsplit\nfrom fastapi import FastAPI, HTTPException\nfrom fastapi.responses import HTMLResponse, FileResponse, JSONResponse\nfrom pydantic import BaseModel, ConfigDict, Field\nfrom core import BASE, RUNS, new_run, read_plan, run_stage\n\nUSER = os.environ.get(\'LAB_USER\', \'owner\')\nPASSWORD = os.environ.get(\'LAB_PASSWORD\', \'\')\nif len(PASSWORD) < 16:\n    raise RuntimeError(\'LAB_PASSWORD 至少 16 字符；使用启动单元格中的隐藏输入\')\napp = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)\njobs = {}\nstate_lock = threading.Lock()\nadmission = threading.Lock()\nFILES = {\'image.png\', \'image_original.png\', \'video.mp4\', \'plan.json\'}\n\nclass Guard:\n    def __init__(self, app):\n        self.app = app\n\n    async def __call__(self, scope, receive, send):\n        if scope[\'type\'] != \'http\':\n            # This app uses polling; no websocket endpoints are needed.\n            if scope[\'type\'] == \'websocket\':\n                await send({\'type\': \'websocket.close\', \'code\': 1008})\n                return\n            return await self.app(scope, receive, send)\n        headers = dict(scope.get(\'headers\', []))\n        authenticated = False\n        try:\n            kind, value = headers.get(b\'authorization\', b\'\').decode().split(\' \', 1)\n            if kind.lower() == \'basic\':\n                user, password = base64.b64decode(value, validate=True).decode(\'utf-8\').split(\':\', 1)\n                u = secrets.compare_digest(user.encode(), USER.encode())\n                p = secrets.compare_digest(password.encode(), PASSWORD.encode())\n                authenticated = u & p\n        except (ValueError, UnicodeError, binascii.Error):\n            pass\n        if not authenticated:\n            response = JSONResponse({\'detail\': \'Authentication required\'}, status_code=401,\n                headers={\'WWW-Authenticate\': \'Basic realm="Wan L4", charset="UTF-8"\',\n                         \'Cache-Control\': \'no-store\'})\n            return await response(scope, receive, send)\n        if scope[\'method\'] in (\'POST\', \'PUT\', \'PATCH\', \'DELETE\'):\n            origin = headers.get(b\'origin\', b\'\').decode()\n            if origin and urlsplit(origin).netloc != headers.get(b\'host\', b\'\').decode():\n                return await JSONResponse({\'detail\': \'Origin rejected\'}, 403)(scope, receive, send)\n            if headers.get(b\'content-type\', b\'\').split(b\';\')[0] != b\'application/json\':\n                return await JSONResponse({\'detail\': \'JSON required\'}, 415)(scope, receive, send)\n            chunks, total = [], 0\n            while True:\n                msg = await receive()\n                if msg[\'type\'] == \'http.disconnect\':\n                    return\n                total += len(msg.get(\'body\', b\'\'))\n                if total > 16384:\n                    return await JSONResponse({\'detail\': \'Body too large\'}, 413)(scope, receive, send)\n                chunks.append(msg.get(\'body\', b\'\'))\n                if not msg.get(\'more_body\', False):\n                    break\n            delivered = False\n            original_receive = receive\n            async def replay():\n                nonlocal delivered\n                if not delivered:\n                    delivered = True\n                    return {\'type\': \'http.request\', \'body\': b\'\'.join(chunks), \'more_body\': False}\n                return await original_receive()\n            receive = replay\n        async def safe_send(message):\n            if message[\'type\'] == \'http.response.start\':\n                message[\'headers\'] = list(message.get(\'headers\', [])) + [\n                    (b\'cache-control\', b\'no-store\'), (b\'x-content-type-options\', b\'nosniff\'),\n                    (b\'x-frame-options\', b\'DENY\'), (b\'referrer-policy\', b\'no-referrer\')]\n            await send(message)\n        await self.app(scope, receive, safe_send)\n\napp.add_middleware(Guard)\n\nclass JobRequest(BaseModel):\n    model_config = ConfigDict(extra=\'forbid\', strict=True)\n    idea: str = Field(min_length=1, max_length=2000)\n    seed: int = Field(default=42, ge=0, le=4294967295)\n    frames: int = 33\n    steps: int = Field(default=20, ge=10, le=50)\n\ndef snapshot(job_id):\n    if not re.fullmatch(r\'[0-9a-f]{32}\', job_id):\n        raise HTTPException(404)\n    with state_lock:\n        if job_id not in jobs:\n            raise HTTPException(404)\n        return json.loads(json.dumps(jobs[job_id]))\n\ndef update(job_id, **changes):\n    with state_lock:\n        jobs[job_id].update(changes)\n\ndef execute(run):\n    job_id = run.name\n    try:\n        for stage in (\'text\', \'image\', \'video\'):\n            update(job_id, stage=stage)\n            run_stage(stage, run)\n            if stage == \'text\':\n                update(job_id, plan=read_plan(run), text_ready=True)\n            elif stage == \'image\':\n                update(job_id, image_ready=True)\n            else:\n                update(job_id, video_ready=True)\n        update(job_id, status=\'done\', stage=\'complete\')\n    except Exception as error:\n        # Full details stay in local logs, no filesystem paths/stack traces in public API.\n        (run / \'error.txt\').write_text(repr(error))\n        update(job_id, status=\'error\', error=\'阶段失败；在 Colab 查看本次运行目录中的日志\')\n    finally:\n        admission.release()\n\n@app.get(\'/\', response_class=HTMLResponse)\ndef home():\n    return (BASE / \'index.html\').read_text()\n\n@app.get(\'/health\')\ndef health():\n    return {\'ok\': True, \'busy\': admission.locked(), \'real_gpu_verified\': False}\n\n@app.post(\'/jobs\', status_code=202)\ndef create_job(body: JobRequest):\n    if body.frames not in (33, 65, 121) or not body.idea.strip():\n        raise HTTPException(422, \'frames 只能为 33/65/121，想法不能为空\')\n    if not admission.acquire(blocking=False):\n        raise HTTPException(409, \'已有任务运行中\')\n    try:\n        with state_lock:\n            if len(jobs) >= 100:\n                raise HTTPException(429, \'本次服务已达 100 次运行，请保存结果并重启服务\')\n        run = new_run(body.idea, body.seed, body.frames, body.steps)\n        with state_lock:\n            jobs[run.name] = dict(id=run.name, status=\'running\', stage=\'text\',\n                text_ready=False, image_ready=False, video_ready=False)\n        threading.Thread(target=execute, args=(run,), daemon=True).start()\n        return {\'id\': run.name, \'status_url\': \'/jobs/\' + run.name}\n    except BaseException:\n        admission.release()\n        raise\n\n@app.get(\'/jobs/{job_id}\')\ndef get_job(job_id: str):\n    return snapshot(job_id)\n\n@app.get(\'/files/{job_id}/{name}\')\ndef get_file(job_id: str, name: str):\n    state = snapshot(job_id)\n    if name not in FILES:\n        raise HTTPException(404)\n    ready = state.get(\'video_ready\') if name == \'video.mp4\' else (\n        state.get(\'text_ready\') if name == \'plan.json\' else state.get(\'image_ready\'))\n    if not ready:\n        raise HTTPException(404)\n    path = RUNS / job_id / name\n    if not path.is_file() or path.is_symlink():\n        raise HTTPException(404)\n    media = \'video/mp4\' if name.endswith(\'.mp4\') else (\n        \'image/png\' if name.endswith(\'.png\') else \'application/json\')\n    return FileResponse(path, media_type=media)\n\nif __name__ == \'__main__\':\n    import uvicorn\n    uvicorn.run(app, host=\'127.0.0.1\', port=int(os.environ.get(\'LAB_PORT\', \'8000\')), access_log=False,\n                proxy_headers=False, workers=1)\n', 'index.html': '<!doctype html><html lang="zh-CN"><meta charset="utf-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<title>想法到短片 · Wan L4</title>\n<style>\nbody{font:16px/1.6 system-ui;background:#f5f3ef;color:#24312e;margin:32px auto;padding:0 20px;max-width:900px}\nh1{font-size:30px}section{background:white;border-radius:16px;padding:24px;margin:20px 0}\ntextarea{box-sizing:border-box;width:100%;padding:12px;font:inherit}button{padding:12px 24px;background:#245f50;color:white;border:0;border-radius:8px;cursor:pointer}\nbutton:disabled{opacity:.5}input,select{padding:8px;margin:8px}pre{white-space:pre-wrap;overflow-wrap:anywhere}img,video{max-width:100%;border-radius:10px}small{color:#53635e}\n</style>\n<h1>想法 → 文案 → 首帧 → 视频</h1>\n<p>Qwen 1.5B · SD-Turbo · 官方 Wan2.2 TI2V 5B。每阶段独立释放显存。</p>\n<section><label for="idea">描述一个单镜头想法</label>\n<textarea id="idea" rows="3" maxlength="2000">一只金毛犬在晨光中的草地上慢慢奔跑，野花轻轻摇动，温暖电影感。</textarea>\n<label>Seed<input id="seed" type="number" min="0" max="4294967295" value="42"></label>\n<label>帧数<select id="frames"><option>33</option><option>65</option><option>121</option></select></label>\n<label>步数<input id="steps" type="number" min="10" max="50" value="20"></label>\n<p><button id="go">生成短片</button> <span id="status">尚未开始</span></p>\n<small>33 帧约 1.38 秒；121 帧约 5.04 秒（24 FPS）。短片生成可能较慢，请保留 Colab 运行连接。</small></section>\n<section><h2>1 · 文案与提示词</h2><pre id="plan">等待生成</pre></section>\n<section><h2>2 · 视频首帧</h2><p>原图为 512×512；此处展示裁切放大的 1280×704 实际输入。</p><img id="image" hidden alt="生成的视频首帧"></section>\n<section><h2>3 · 视频结果</h2><video id="video" hidden controls playsinline preload="metadata"></video><p><a id="download" hidden>下载 MP4</a></p></section>\n<script>\nconst $=id=>document.getElementById(id); let timer=null,current=null;\nasync function get(url,options){const r=await fetch(url,options);if(!r.ok)throw Error(\'HTTP \'+r.status+\': \'+await r.text());return r.json()}\nasync function poll(){try{const j=await get(\'/jobs/\'+current);$(\'status\').textContent=j.status+\' · \'+j.stage;\nif(j.text_ready)$(\'plan\').textContent=JSON.stringify(j.plan,null,2);\nif(j.image_ready&&$(\'image\').hidden){$(\'image\').src=\'/files/\'+current+\'/image.png\';$(\'image\').hidden=false}\nif(j.video_ready&&$(\'video\').hidden){const u=\'/files/\'+current+\'/video.mp4\';$(\'video\').src=u;$(\'video\').hidden=false;$(\'download\').href=u;$(\'download\').download=\'wan-video.mp4\';$(\'download\').hidden=false}\nif(j.status===\'error\')$(\'status\').textContent=j.error;\nif(j.status===\'running\'){timer=setTimeout(poll,4000)}else{$(\'go\').disabled=false;sessionStorage.removeItem(\'wan_job\')}\n}catch(e){$(\'status\').textContent=\'连接中断，可稍后刷新重试：\'+e.message;$(\'go\').disabled=false}}\n$(\'go\').onclick=async()=>{clearTimeout(timer);$(\'go\').disabled=true;$(\'image\').hidden=true;$(\'video\').hidden=true;$(\'video\').removeAttribute(\'src\');$(\'download\').hidden=true;$(\'plan\').textContent=\'等待生成\';\ntry{const j=await get(\'/jobs\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({idea:$(\'idea\').value,seed:Number($(\'seed\').value),frames:Number($(\'frames\').value),steps:Number($(\'steps\').value)})});current=j.id;sessionStorage.setItem(\'wan_job\',current);poll()}\ncatch(e){$(\'status\').textContent=e.message;$(\'go\').disabled=false}};\ncurrent=sessionStorage.getItem(\'wan_job\');if(current){$(\'go\').disabled=true;poll()}\n</script></html>\n', 'preflight.py': "import json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\ndef host_check(base, before_install=False):\n    if sys.platform != 'linux':\n        raise RuntimeError('请在 Linux Google Colab GPU 运行时执行')\n    rows = subprocess.check_output(['nvidia-smi',\n        '--query-gpu=name,memory.total,memory.free,driver_version',\n        '--format=csv,noheader,nounits'], text=True).strip().splitlines()\n    name, total, free, driver = [x.strip() for x in rows[0].split(',')]\n    if 'L4' not in name:\n        raise RuntimeError('本笔记本针对 NVIDIA L4；当前为 ' + name)\n    if float(total) < 22000 or float(free) < 21000:\n        raise RuntimeError('显存不足或被占用，请用空闲 L4 24GB 运行时')\n    if tuple(map(int, driver.split('.')[:2])) < (550, 54):\n        raise RuntimeError('此固定 CUDA 12.4 环境要求驱动至少 550.54')\n    mem = {s.split(':')[0]: int(s.split()[1])*1024 for s in\n           Path('/proc/meminfo').read_text().splitlines()}\n    available = mem['MemAvailable']/2**30\n    disk = shutil.disk_usage(base).free/2**30\n    if available < 40:\n        raise RuntimeError(f'可用系统内存只有 {available:.1f} GiB；需高内存运行时，建议总内存 64GB')\n    if before_install and disk < 100:\n        raise RuntimeError(f'可用硬盘仅 {disk:.1f} GiB；安装前建议至少 100 GiB')\n    print(json.dumps(dict(gpu=name, vram_total_mib=total, vram_free_mib=free,\n        driver=driver, available_ram_gib=round(available,1), free_disk_gib=round(disk,1)), indent=2))\n\ndef gpu_check():\n    import torch\n    import flash_attn\n    import wan\n    from wan.configs import WAN_CONFIGS, SUPPORTED_SIZES\n    assert torch.__version__.startswith('2.6.0')\n    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n    assert torch._C._GLIBCXX_USE_CXX11_ABI is False, 'FlashAttention wheel ABI 不匹配'\n    from flash_attn import flash_attn_func\n    q = torch.randn(1, 32, 4, 64, device='cuda', dtype=torch.bfloat16)\n    y = flash_attn_func(q, q, q)\n    assert y.shape == q.shape and torch.isfinite(y).all()\n    assert '1280*704' in SUPPORTED_SIZES['ti2v-5B']\n    cfg = WAN_CONFIGS['ti2v-5B']\n    assert (cfg.sample_fps, cfg.sample_steps, cfg.frame_num) == (24, 50, 121)\n    print('CUDA、BF16、FlashAttention2 实际小算子、Wan 导入及任务配置检查通过')\n\nif __name__ == '__main__':\n    base = Path(os.environ.get('WAN_LAB_HOME', '/content/wan_l4_lab'))\n    host_check(base)\n    gpu_check()\n", 'download_models.py': "import json\nimport os\nfrom fnmatch import fnmatch\nfrom pathlib import Path\nimport shutil\nfrom core import BASE\n\n# Set before importing Hugging Face/Xet so cache stays with this Colab project.\nos.environ.setdefault('HF_XET_CACHE', str(BASE / 'hf_cache' / 'xet'))\nfrom huggingface_hub import HfApi, snapshot_download, try_to_load_from_cache\nfrom huggingface_hub.utils._runtime import is_xet_available\n\nIGNORE = {'image': ['sd_turbo.safetensors', 'text_encoder/model.safetensors',\n    'unet/diffusion_pytorch_model.safetensors', 'vae/diffusion_pytorch_model.safetensors']}\n\ndef wanted(name, filename):\n    return not any(fnmatch(filename, pattern) for pattern in IGNORE.get(name, []))\n\ndef download():\n    if not is_xet_available():\n        raise RuntimeError('下载环境缺少 hf-xet；在独立环境安装 hf-xet==1.8.0 后重跑第 4 节')\n    from hf_xet import PyPointerFile, download_files\n    print('Xet 客户端可用；使用 4 个文件下载线程。实际速度取决于网络与磁盘。', flush=True)\n    cfg = json.loads((BASE / 'config.json').read_text())\n    api = HfApi()\n    models, manifests = {}, {}\n    remaining = 0\n    for name, item in cfg['model_specs'].items():\n        info = api.model_info(item['repo'], revision=item['revision'], files_metadata=True)\n        if info.sha != item['revision']:\n            raise RuntimeError('模型版本不一致')\n        manifests[name] = {f.rfilename: f.size for f in info.siblings\n                          if f.size is not None and wanted(name, f.rfilename)}\n        for relative, size in manifests[name].items():\n            cached = try_to_load_from_cache(item['repo'], relative,\n                revision=item['revision'], cache_dir=BASE / 'hf_cache')\n            if isinstance(cached, str) and Path(cached).is_file() and Path(cached).stat().st_size == size:\n                continue\n            remaining += size\n    # Completed cached files are reused. Incomplete files are conservatively counted in full.\n    if shutil.disk_usage(BASE).free < remaining + 15*2**30:\n        raise RuntimeError(f'下载前空间检查失败：尚需下载约 {remaining/2**30:.1f} GiB，另留 15 GiB')\n    for name, item in cfg['model_specs'].items():\n        folder = Path(snapshot_download(item['repo'], revision=item['revision'],\n            cache_dir=BASE / 'hf_cache', max_workers=4,\n            ignore_patterns=IGNORE.get(name)))\n        for relative, size in manifests[name].items():\n            p = folder / relative\n            if not p.is_file() or p.stat().st_size != size:\n                raise RuntimeError('下载文件缺失或长度不符：' + relative)\n        models[name] = str(folder)\n        print(name, '固定版本下载及文件长度检查通过', flush=True)\n    cfg['models'] = models\n    (BASE / 'config.json').write_text(json.dumps(cfg, ensure_ascii=False, indent=2))\n    (BASE / 'model_manifest.json').write_text(json.dumps(manifests, indent=2))\n    # Check official format before allocating models.\n    wan = Path(models['wan'])\n    for rel in ['config.json', 'Wan2.2_VAE.pth', 'models_t5_umt5-xxl-enc-bf16.pth', 'google/umt5-xxl']:\n        if not (wan / rel).exists():\n            raise RuntimeError('官方 Wan 目录缺少：' + rel)\n    print('模型已就绪；后续生成阶段强制离线加载，不调用付费服务。')\n\nif __name__ == '__main__':\n    download()\n", 'requirements.txt': '# Python 3.11; torch/torchvision installed separately from official cu124 index.\ntorch==2.6.0\ntorchvision==0.21.0\ndiffusers==0.33.1\ntransformers==4.51.3\ntokenizers==0.21.1\naccelerate==1.6.0\nhuggingface-hub==0.30.2\nhf-xet==1.8.0\nsafetensors==0.5.3\nnumpy==1.26.4\nopencv-python==4.11.0.86\nPillow==11.2.1\nimageio==2.37.0\nimageio-ffmpeg==0.6.0\neinops==0.8.1\neasydict==1.13\nftfy==6.3.1\ndashscope==1.23.2\ntqdm==4.67.1\nfastapi==0.115.12\nstarlette==0.46.2\npydantic==2.11.4\nuvicorn==0.34.2\nhttpx==0.28.1\npackaging==25.0\n', 'tunnel_runtime.py': "# Cloudflare recovery cell: only restarts this notebook's tunnel, never the model server.\nimport base64\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport signal\nimport subprocess\nimport time\nimport urllib.request\nimport urllib.error\n\nCF_SHA256 = '56f13335143ec0e22772963290683a84a04114a35ec64cbc794a4d39ff9bb8cf'\nCF_BINARY_URL = 'https://github.com/cloudflare/cloudflared/releases/download/2026.11.2/cloudflared-linux-amd64'\n\nclass NoRedirect(urllib.request.HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        return None  # Never forward credentials to a redirected host.\n\ndef probe_health(url, auth=None, timeout=5):\n    headers = {'Accept': 'application/json', 'User-Agent': 'Wan-L4-healthcheck/2'}\n    if auth:\n        headers['Authorization'] = 'Basic ' + base64.b64encode(\n            (auth[0] + ':' + auth[1]).encode()).decode()\n    request = urllib.request.Request(url.rstrip('/') + '/health', headers=headers)\n    try:\n        with urllib.request.build_opener(NoRedirect()).open(request, timeout=timeout) as response:\n            code, body = response.status, response.read(4096)\n            challenge = response.headers.get('WWW-Authenticate', '')\n    except urllib.error.HTTPError as error:\n        code, body = error.code, error.read(4096)\n        challenge = error.headers.get('WWW-Authenticate', '')\n    except (OSError, urllib.error.URLError) as error:\n        return {'code': None, 'ok': False, 'app401': False,\n                'detail': type(error).__name__ + ': ' + str(error)}\n    try:\n        data = json.loads(body)\n    except (ValueError, UnicodeError):\n        data = {}\n    return {'code': code, 'ok': code == 200 and isinstance(data, dict) and data.get('ok') is True,\n        'app401': code == 401 and 'Basic' in challenge and isinstance(data, dict)\n                  and data.get('detail') == 'Authentication required',\n        'detail': 'HTTP ' + str(code)}\n\ndef classify_probes(anonymous, authenticated):\n    if anonymous['code'] is not None and 200 <= anonymous['code'] < 300:\n        return 'unsafe', '匿名请求意外成功，不能公开该服务'\n    if anonymous['app401'] and authenticated['ok']:\n        return 'ready', '匿名 401 / 带密码 200，且健康响应内容正确'\n    if anonymous['app401'] and authenticated['code'] == 401:\n        return 'auth', '已到达应用，但带密码请求仍为 401：重跑第 9 节以同步服务密码与 WEB_AUTH'\n    return 'retry', '匿名=' + anonymous['detail'] + '；带密码=' + authenticated['detail']\n\ndef stop_tunnel(proc):\n    if proc is not None and proc.poll() is None:\n        os.killpg(proc.pid, signal.SIGTERM)\n        try:\n            proc.wait(timeout=10)\n        except subprocess.TimeoutExpired:\n            os.killpg(proc.pid, signal.SIGKILL)\n            proc.wait()\n\ndef recover_tunnel(base, auth, old_tunnel=None, seconds_per_protocol=120, origin='http://127.0.0.1:8000'):\n    base = Path(base)\n    if not auth or len(auth) != 2:\n        raise RuntimeError('找不到 WEB_AUTH，请先运行第 9 节启动带密码服务')\n    # Validate the origin before creating any public exposure.\n    local_anonymous = probe_health(origin)\n    local_authenticated = probe_health(origin, auth)\n    status, detail = classify_probes(local_anonymous, local_authenticated)\n    print('本地自检：', detail, flush=True)\n    if status != 'ready':\n        raise RuntimeError('本地服务未通过检查；先处理第 9 节。' + detail)\n    binary = base / 'cloudflared'\n    if not binary.exists() or hashlib.sha256(binary.read_bytes()).hexdigest() != CF_SHA256:\n        temp = binary.with_suffix('.download')\n        urllib.request.urlretrieve(CF_BINARY_URL, temp)\n        if hashlib.sha256(temp.read_bytes()).hexdigest() != CF_SHA256:\n            temp.unlink()\n            raise RuntimeError('cloudflared SHA256 不匹配')\n        temp.replace(binary)\n    binary.chmod(0o700)\n    if any((Path.home()/'.cloudflared'/name).exists() for name in ('config.yml', 'config.yaml')):\n        raise RuntimeError('存在 cloudflared 配置，请在干净运行时处理，避免使用已有隧道配置')\n    stop_tunnel(old_tunnel)\n    last_detail = '没有获取到临时地址'\n    # Preserve auto (QUIC with built-in fallback); HTTP/2 is a second diagnostic attempt.\n    for protocol in ('auto', 'http2'):\n        log_path = base / ('tunnel_' + protocol + '.log')\n        print('启动 Cloudflare，协议=' + protocol + '；日志=' + str(log_path), flush=True)\n        command = [str(binary), 'tunnel', '--no-autoupdate', '--protocol', protocol,\n            '--url', origin]\n        with log_path.open('w') as log:\n            proc = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,\n                                    start_new_session=True)\n        success = False\n        url, previous_detail = None, None\n        deadline = time.monotonic() + seconds_per_protocol\n        try:\n            while time.monotonic() < deadline:\n                if proc.poll() is not None:\n                    last_detail = 'cloudflared 已退出，exit=' + str(proc.returncode)\n                    break\n                log_text = log_path.read_text(errors='replace')\n                match = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', log_text)\n                if match:\n                    url = match.group(0)\n                    # A printed URL is not proof that the tunnel is connected.\n                    anonymous = probe_health(url)\n                    authenticated = probe_health(url, auth)\n                    status, last_detail = classify_probes(anonymous, authenticated)\n                    if status == 'ready':\n                        success = True\n                        print('公网自检通过：' + last_detail, flush=True)\n                        return proc, url\n                    if status in ('unsafe', 'auth'):\n                        raise RuntimeError(last_detail)\n                    if last_detail != previous_detail:\n                        print('等待隧道/DNS就绪：' + last_detail, flush=True)\n                        previous_detail = last_detail\n                time.sleep(2)\n            print('本次未就绪：' + last_detail, flush=True)\n            # Keep full logs local; users can share the relevant error lines without credentials.\n            print('请查看日志最后约 30 行：' + str(log_path), flush=True)\n        finally:\n            if not success:\n                stop_tunnel(proc)\n    raise RuntimeError('公网未就绪：' + last_detail +\n        '。已停止失败隧道，本地服务和模型结果仍保留；查看 tunnel_http2.log / tunnel_auto.log。')\n\n", 'oneclick_control.py': '"""Standard-library launcher; credentials and process handles stay in kernel memory."""\nimport os\nfrom pathlib import Path\nimport secrets\nimport signal\nimport socket\nimport subprocess\nimport time\nfrom tunnel_runtime import probe_health, classify_probes, recover_tunnel, stop_tunnel\n\ndef alive(process):\n    return process is not None and process.poll() is None\n\ndef origin_ready(state):\n    if not alive(state.get(\'server\')) or not state.get(\'auth\') or not state.get(\'origin\'):\n        return False\n    status, _ = classify_probes(probe_health(state[\'origin\']),\n                               probe_health(state[\'origin\'], state[\'auth\']))\n    return status == \'ready\'\n\ndef stop_oneclick_state(state):\n    """Stop only this launcher\'s owned server/tunnel and matching model workers."""\n    stop_tunnel(state.get(\'tunnel\'))\n    stop_tunnel(state.get(\'server\'))\n    # Workers use their own process groups. Match the exact server-owned run paths.\n    base = Path(state.get(\'base\', \'/content/wan_l4_lab\')).resolve()\n    if state.get(\'server\') is not None:\n        for proc in Path(\'/proc\').iterdir():\n            if not proc.name.isdigit():\n                continue\n            try:\n                args = [v.decode(errors=\'replace\') for v in\n                        (proc/\'cmdline\').read_bytes().split(b\'\\0\') if v]\n                worker = str(base/\'worker.py\') in args or str(base/\'Wan2.2/generate.py\') in args\n                owned_run = any(v.startswith(str(base/\'runs\')+\'/\') for v in args)\n                pid = int(proc.name)\n                if worker and owned_run and os.getpgid(pid) == pid:\n                    os.killpg(pid, signal.SIGTERM)\n                    time.sleep(0.2)\n                    try:\n                        os.killpg(pid, signal.SIGKILL)\n                    except ProcessLookupError:\n                        pass\n            except (FileNotFoundError, ProcessLookupError, PermissionError):\n                pass\n    state.clear()\n\ndef start_oneclick(base, python, state):\n    base = Path(base).resolve()\n    if not origin_ready(state):\n        # Never kill an unrelated server using port 8000. Allocate a separate local port.\n        stop_tunnel(state.get(\'tunnel\'))\n        stop_tunnel(state.get(\'server\'))\n        with socket.socket() as sock:\n            sock.bind((\'127.0.0.1\', 0))\n            port = sock.getsockname()[1]\n        auth = (\'owner\', secrets.token_urlsafe(24))\n        state.clear()\n        state.update(base=str(base), auth=auth, origin=f\'http://127.0.0.1:{port}\',\n                     server=None, tunnel=None, url=None)\n        environment = dict(os.environ, LAB_USER=auth[0], LAB_PASSWORD=auth[1], LAB_PORT=str(port))\n        # Credentials are passed only to our server, not written to a file or command arguments.\n        with (base/\'oneclick_server.log\').open(\'w\') as log:\n            try:\n                state[\'server\'] = subprocess.Popen([str(python), str(base/\'server.py\')],\n                    cwd=base, env=environment, stdout=log, stderr=subprocess.STDOUT,\n                    start_new_session=True)\n            finally:\n                environment.pop(\'LAB_PASSWORD\', None)\n    try:\n        deadline = time.monotonic() + 60\n        while time.monotonic() < deadline:\n            if not alive(state[\'server\']):\n                raise RuntimeError(\'网页服务退出；查看 oneclick_server.log\')\n            if origin_ready(state):\n                break\n            time.sleep(0.5)\n        else:\n            raise RuntimeError(\'本地网页服务未通过 401/200 自检；查看 oneclick_server.log\')\n        if alive(state.get(\'tunnel\')) and state.get(\'url\'):\n            status, _ = classify_probes(probe_health(state[\'url\']),\n                                       probe_health(state[\'url\'], state[\'auth\']))\n            if status == \'ready\':\n                return state\n        old_tunnel = state.get(\'tunnel\')\n        state[\'url\'], state[\'tunnel\'] = None, None\n        state[\'tunnel\'], state[\'url\'] = recover_tunnel(base, state[\'auth\'],\n            old_tunnel, origin=state[\'origin\'])\n        return state\n    except BaseException:\n        stop_tunnel(state.get(\'tunnel\'))\n        stop_tunnel(state.get(\'server\'))\n        state[\'server\'], state[\'tunnel\'], state[\'url\'] = None, None, None\n        raise\n'}
for name, content in FILES.items():
    (BASE / name).write_text(content)
# Reuse only model weights; credentials, server configuration and outputs stay separate.
classic_cache = Path('/content/wan_l4_lab/hf_cache')
if classic_cache.is_dir() and not (BASE/'hf_cache').exists():
    (BASE/'hf_cache').symlink_to(classic_cache, target_is_directory=True)
if str(BASE) not in sys.path: sys.path.insert(0, str(BASE))
from oneclick_control import origin_ready, start_oneclick, stop_oneclick_state
from preflight import host_check
REUSE_SERVER = origin_ready(ONECLICK_STATE)
if REUSE_SERVER:
    print('检测到本一键版网页仍在运行，将复用当前服务与密码，跳过重复安装下载。')
else:
    host_check(BASE, before_install=not (BASE/'venv/bin/python').exists())
PY, REPO = BASE/'venv/bin/python', BASE/'Wan2.2'


## 2 · 自动安装和 GPU 自检
独立 Python 3.11、固定 CUDA/PyTorch、官方预编译 FlashAttention，不替换主内核的 PyTorch。
固定版本减少接口变化；首次需要访问 GitHub、PyPI 和 PyTorch 下载源。


In [ ]:
if not REUSE_SERVER:
    WAN_COMMIT = 'e5058c08fada388b04bb3d1a4734bddbd2fdeac5'
    FLASH_WHEEL = 'https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1%2Bcu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl'
    PY = BASE / 'venv/bin/python'
    REPO = BASE / 'Wan2.2'
    # 仅在 Colab 主环境安装引导器；模型依赖全部放入独立环境。
    run_logged([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                'uv==0.6.17'], 'bootstrap')
    if not PY.exists():
        run_logged([sys.executable, '-m', 'uv', 'venv', '--python', '3.11.11',
                    '--seed', BASE / 'venv'], 'venv')
    run_logged([PY, '-m', 'pip', 'install', 'torch==2.6.0', 'torchvision==0.21.0',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], 'install_torch')
    run_logged([PY, '-m', 'pip', 'install', '-r', BASE / 'requirements.txt'], 'install_deps')
    run_logged([PY, '-m', 'pip', 'install', '--no-deps', FLASH_WHEEL], 'install_flash')
    run_logged([PY, '-m', 'pip', 'check'], 'pip_check')
    if not shutil.which('ffprobe'):
        run_logged(['apt-get', 'update', '-qq'], 'apt_update')
        run_logged(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], 'apt_ffmpeg')
    if not REPO.exists():
        run_logged(['git', 'clone', '--filter=blob:none', '--no-checkout',
                    'https://github.com/Wan-Video/Wan2.2.git', REPO], 'clone_wan')
    if subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() != 'https://github.com/Wan-Video/Wan2.2.git':
        raise RuntimeError('源码目录的来源不匹配，请换干净运行时')
    run_logged(['git', '-C', REPO, 'checkout', '--detach', WAN_COMMIT], 'checkout_wan')
    assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == WAN_COMMIT
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('官方源码存在本地修改，请使用干净目录')
    (BASE / 'environment.freeze.txt').write_text(subprocess.check_output([str(PY), '-m', 'pip', 'freeze'], text=True))
    cfg = dict(wan_commit=WAN_COMMIT, verified_on='2026-11-26',
               model_specs={'wan': {'repo': 'Wan-AI/Wan2.2-TI2V-5B', 'revision': '921dbaf3f1674a56f47e83fb80a34bac8a8f203e'}, 'text': {'repo': 'Qwen/Qwen2.5-1.5B-Instruct', 'revision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'}, 'image': {'repo': 'stabilityai/sd-turbo', 'revision': 'b261bac6fd2cf515557d5d0707481eafa0485ec2'}}, models={})
    # 重新安装环境后，重新运行下载单元格以核对并恢复模型目录。
    (BASE / 'config.json').write_text(json.dumps(cfg, ensure_ascii=False, indent=2))
    print('安装完成。无需重启 Colab 主内核。')

else:
    print("服务已就绪，跳过安装。")


In [ ]:
if not REUSE_SERVER:
    env = dict(os.environ, PYTHONPATH=str(REPO)+os.pathsep+str(BASE))
    run_logged([PY, BASE / 'preflight.py'], 'gpu_selfcheck', env=env, cwd=REPO)
    help_text = subprocess.check_output([str(PY), str(REPO/'generate.py'), '--help'],
                                       cwd=REPO, env=env, text=True)
    for flag in ['--task', '--image', '--frame_num', '--save_file', '--offload_model',
                 '--convert_model_dtype', '--t5_cpu', '--sample_steps', '--sample_solver']:
        assert flag in help_text, '官方 CLI 缺少参数 '+flag
    (BASE / 'generate-help.txt').write_text(help_text)
    print('官方 CLI 参数检查通过。')

else:
    print("复用已启动服务，不打断正在运行的 GPU 任务。")


## 3 · 自动下载模型
使用已安装的 Xet 客户端和 4 个文件下载线程；只下载工作流需要的 SD-Turbo FP16 权重。
复用已完成缓存，不需要登录。默认使用公开模型，不读取会触发授权提示的 Colab 密钥。
若你的网络或 Hugging Face 对匿名访问限流，会显示具体下载错误，需要按错误处理后重跑。


In [ ]:
if not REUSE_SERVER:
    run_logged([PY, '-m', 'pip', 'install', 'hf-xet==1.8.0'], 'install_xet')
    download_env = dict(os.environ)
    download_env.pop('HF_HUB_DISABLE_XET', None)
    download_env['HF_XET_CACHE'] = str(BASE/'hf_cache'/'xet')
    # Existing environment tokens are optional; no userdata.get/input/getpass prompt.
    try:
        run_logged([PY, BASE/'download_models.py'], 'download_models', env=download_env)
    finally:
        download_env.pop('HF_TOKEN', None)
    (BASE/'environment.freeze.txt').write_text(
        subprocess.check_output([str(PY), '-m', 'pip', 'freeze'], text=True))
else:
    print('复用已下载并启动的模型配置。')
configuration = json.loads((BASE/'config.json').read_text())
if set(configuration.get('models', {})) != {'text', 'image', 'wan'}:
    raise RuntimeError('模型未就绪，不能启动公网服务；查看 download_models.log')
for folder in configuration['models'].values():
    if not Path(folder).is_dir():
        raise RuntimeError('模型缓存目录丢失，请重跑模型下载')
BOOT_READY = True
print('模型已准备好。安装/下载完成；下一步自动创建账号密码并验证公网。')


## 4 · 自动启动并显示访问凭据
默认创建临时公网入口。强密码使用系统安全随机源，账号为 `owner`。
先检查本地匿名 401 / 带密码健康 200，再检查公网；auto、HTTP/2 各最多等待 120 秒。
只有通过自检才会显示下面的地址和密码。启动时不会自动生成视频；请进入网页提交任务。

运行完成后保留此 Colab 连接。需要主动停止时，在新单元格运行 `stop_oneclick()`；它只关闭本一键版的进程。


In [ ]:
if not BOOT_READY:
    raise RuntimeError('前面的安装、GPU 检查或模型下载尚未完成')
from IPython.display import HTML
from html import escape

def stop_oneclick():
    stop_oneclick_state(ONECLICK_STATE)
    print('一键版网页、公网入口及匹配的生成进程已停止；模型和结果文件仍保留。')

try:
    start_oneclick(BASE, PY, ONECLICK_STATE)
except BaseException:
    print('启动未完成：查看上方具体错误，以及 oneclick_server.log、tunnel_auto.log、tunnel_http2.log。')
    print('模型缓存与已有产物仍保留，可修复问题后再次运行此单元格或全部运行。')
    raise
SERVER = ONECLICK_STATE['server']
TUNNEL = ONECLICK_STATE['tunnel']
PUBLIC_URL = ONECLICK_STATE['url']
WEB_AUTH = ONECLICK_STATE['auth']
assert PUBLIC_URL.startswith('https://')
# Never save this card, password or authenticated URL to a file.
print('✅ 一键启动完成：公网鉴权自检通过。')
print('公网地址：', PUBLIC_URL)
print('账号：', WEB_AUTH[0])
print('密码：', WEB_AUTH[1])
display(HTML(
    '<div style="padding:24px;border:2px solid #245f50;border-radius:14px;line-height:1.8">'
    '<h2>✅ 一键版已启动</h2>'
    '<p><a target="_blank" rel="noopener noreferrer" href="'+escape(PUBLIC_URL, quote=True)+'">'
    '点击打开公网创作页面</a></p>'
    '<p>账号：<code>'+escape(WEB_AUTH[0])+'</code><br>'
    '密码：<code style="user-select:all">'+escape(WEB_AUTH[1])+'</code></p>'
    '<p>进入网页后输入想法并点击「生成短片」。请保存凭据，分享笔记本前清除输出。</p></div>'
))
